# Python for Neuroscience (2026)

# 01. Patch-clamp: passive membrane properties

<!-- <img src="Images/patch-clamp_brain_slices.png" width="800"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/patch-clamp_brain_slices.png" width="800">

**GOAL**: How to calculate some basic passive membrane properties from current clamp recordings using custom functions.

**Further reading**
- [A guide to Passive Membrane Properties](https://www.scientifica.uk.com/learning-zone/passive-membrane-properties). Scientifica.

**Further Python resources**
- [Google Python Style Guide](https://google.github.io/styleguide/pyguide.html).
- [PEP 8 – Style Guide for Python Code](https://peps.python.org/pep-0008/).
- [PEP 20 – The Zen of Python](https://peps.python.org/pep-0020/).

# Import all the necessary libraries

**Tips**:
- Check the official documentation to understand how a library works. Most widely used libraries are well documented.
- Use help() in Python for quick information about functions and modules.

In [ ]:
!pip install pyabf

In [ ]:
# Built-in model to interact with the operation system
import os

# To import and handle patch clamp abf files
import pyabf

# For arrays and matrices
import numpy as np

# Dataframes
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Scipy for stats and curve fitting
from scipy.stats import linregress
from scipy.optimize import curve_fit

# Define the paths to folders and files

Defining paths is handy for accessing and saving data consistently.

[`os.path`](https://docs.python.org/3/library/os.path.html). Python module for path names.

- Absolute path is the complete path to a file or folder.
- Relative path refers is the path relative to the current working directory.

Read more about [relative and absolute paths](https://automatetheboringstuff.com/1e/chapter8/).  

**Data management: Resources**
- [FAIR Guiding Principles for scientific data management and stewardship](https://www.go-fair.org/fair-principles/)
- [NIN guidelines for data storage](https://nin.nl/wp-content/uploads/sites/2/2024/10/data-storage-protocol-NIN_20211210.pdf).
- [NIH data management](https://grants.nih.gov/policy-and-compliance/policy-topics/sharing-policies/dms/data-management).
- [Cambridge research data management](https://www.data.cam.ac.uk/about).

In [ ]:
# Create a folder name for the patch-clamp notebooks
notebook_name = 'patch-clamp'

# Get the directory where the data are located
data_path = os.getcwd()

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Example Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Download the data

In [ ]:
if 'google.colab' in str(get_ipython()):
    # -O tells wget where to save the file locally
    !wget -O "Example Data/pfc_pyr_passive.abf" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/pfc_pyr_passive.abf"
else:
    print("Download data from 'Example Data' folder.")

# Load the data

<!-- <img src="Images/01_membrane-properties_protocol.png" width="400"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/01_membrane-properties_protocol.png" width="400">

**Example data**: Whole-cell current-clamp recording from a pyramidal neuron in L2/3 of the mouse PFC. Small current steps near the resting membrane potential were used to measure passive membrane properties.
- Filename: **pfc_pyr_passive.abf**.

This is a recording from the path-clamp software Clampex (pClamp) used in many laboratories.

To read the file, Scott Harden created the library [**pyabf**](https://swharden.com/pyabf/) for reading electrophysiology data from Axon Binary Format (ABF) files in Python. pyABF is pure Python and only has matplotlib and numpy as dependencies. See the [documentation](https://swharden.com/pyabf/tutorial/) for the commands.

In [ ]:
# ABF file/s
filename = "pfc_pyr_passive"

# For compatability between OS, os.path.join() is better
# data_path = os.path.join(paths['data'], f"{filename}.abf")

# ... but I started using used strings fore readibility
data_path = f"{paths['data']}/{filename}.abf"  # strings or use
abf = pyabf.ABF(data_path)
print(abf)  # Print information about the file

# print(abf.headerText) # display header information in the console
# abf.headerLaunch() # display header information in a web browser

## Q: How can you inspect one specific function in a library?

In Python, a function is a reusable block of code defined with def. A method is a function that is defined inside a class and operates on instances of that class.

Read more about [classess](https://docs.python.org/3/tutorial/classes.html).

**Tip**: You can explore the contents of a library (such as its classes, functions, attributes, and methods) using built-in tools like dir() and help().

<details>
<summary><strong>Solution</strong></summary>

```python

import pyabf

# List all attributes and methods of the ABF class
dir(pyabf.ABF)

# Documentation for that class
# help(pyabf.ABF)

# Select one method to see help/documentation
# help(pyabf.ABF.sweepC)
```

# Quick plot

In **pyABF**, each trace is called a **sweep**, and it is helpful to follow this nomenclature in your code for clarity. For example, `abf.sweepList` contains the list of all sweeps (traces) in the recording.

For current-clamp recordings, plot voltage in y-axis, and time in x-axis, which corresponds to `abf.sweepY` and `abf.sweepX`, respectively. **Important**, the order in plt.plot is always: x, y.

There are two main ways to create plots in Python:
- `plt.figure()` follows the MATLAB-style.
- `fig, ax` uses Python’s object-oriented approach. Here, you explicitly create figure and axes objects and call plotting methods on the axes. More flexibility.

In [ ]:
# Quick plot to see the trace/s
plt.figure(figsize=(6,3))  # You can define here the size

for sweepNumber in abf.sweepList:  # Loop through all traces in the recording
    abf.setSweep(sweepNumber)  # sweep number to load
    plt.plot(abf.sweepX, abf.sweepY)
    plt.ylabel(abf.sweepLabelY)  # This abf method automatically retrieve labels
    plt.xlabel(abf.sweepLabelX)

plt.show()

## Q: Plot only one trace of the sweeps

**Tip**: remove loop and refine `abf.setSweep(sweepNumber)`, or keep loop and add `sweepNumber == trace`, or set a range in `abf.sweepList`. Which method is more efficient?

In [ ]:
plt.figure(figsize=(8,4))

trace = 4  # Important: Python indices start at 0

for sweepNumber in abf.sweepList:
    # COMPLETE THE CODE


plt.show()

<details>
<summary><strong>Solution</strong></summary>

```python

# Quick plot to see the trace/s
plt.figure(figsize=(8,4))

trace = 4  # Important: Python indices start at 0

for sweepNumber in abf.sweepList[4:5]:  # or abf.sweepList[trace:trace]
    # if sweepNumber == trace:
    abf.setSweep(sweepNumber)
    plt.plot(abf.sweepX, abf.sweepY)
    plt.ylabel(abf.sweepLabelY)
    plt.xlabel(abf.sweepLabelX)

plt.show()
```

# Function passive properties: RMP

**RESTING MEMBRANE POTENTIAL** (RMP) is the steady-state voltage of the cell when no current is applied. Generally, it is between -60 and -80 mV.

The membrane potential is defined as the potential of the inner side of the membrane relative to the potential of the outer side, given by the [Goldman-Hodgkin-Katz Equation](https://www.physiologyweb.com/calculators/ghk_equation_calculator.html). In other words, it is the potential difference across the membrane, which depends on the intracellular and extracellular concentrations of ions, mainly potassium, as well as their permeabilities.

We will now create a simple function to calculate the resting membrane potential by averaging the initial segment of the trace when I = 0. Functions are useful for reusing code, especially for longer scripts.

First, we need to define the beginning and end of the segment over which to calculate the average.

- The output of the function is a **dictionary**, where you can get a value using its name. For example: `features['rmp']`.  
- Dictionaries are similar to **lists** `[]`, but instead of indices (positions), dictionaries use **keys** to access values.  
- Tuples `()` are similar to lists, but they cannot be changed after they are created. Like lists, items in tuples are accessed by position, not by name.

**More about functions**:

- [Creating functions. Software Carpentry](https://swcarpentry.github.io/python-novice-inflammation/instructor/08-func.html).

In [ ]:
def passive_memb_properties(voltage_signal,
                            fs,
                            cursor1_ms=5,  # default cursor positions
                            cursor2_ms=50):  # default cursor positions

    # Convert cursor position in ms indices (depends on sampling frequency)
    cursor1 = int(cursor1_ms * (fs/1000))
    cursor2 = int(cursor2_ms * (fs/1000))

    # Calculate RMP at the between of the trace when I = 0
    rmp = np.average(voltage_signal[cursor1:cursor2])

    # Return results as dictionary
    return {'rmp': rmp}

## Plot and table

## Q: Check if the RMP values in the table look correct compared to the plot

**Tips**: Even simple plots are a great method to check if the results look correct.

In [ ]:
# Sampling frequency (fs)
fs = int(abf.dataPointsPerMs * 1000)

# Define channels for each signal
voltage_channel = 0
current_channel = 0

# WE CREATE A TABLE FOR THE RESULTS
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.3f}'.format

# Calculate the results for each sweep
for sweep in abf.sweepList:

    # Define channels and signals for the function
    abf.setSweep(sweep, channel=1)
    voltage_signal = abf.sweepY
    current_signal = abf.sweepC

    # HERE WE CALL THE FUNCTION
    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       fs=fs, cursor1_ms=5, cursor2_ms=500)

    # FILL THE TABLE WITH RESULTS FROM THE FUNCTION
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']

# Create a list the position of cursors in ms
cursors_ms = [5, 100]

# Quick plot to visualize the plot and the cursors position
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=0)
    ax1.plot(abf.sweepX*1000, abf.sweepY)

    abf.setSweep(sweep, channel=0)
    ax2.plot(abf.sweepX*1000, abf.sweepC)

# Labels
ax1.set_ylabel(abf.sweepLabelY)
ax2.set_ylabel(abf.sweepLabelC)
ax2.set_xlabel('Time (ms)')

# Plot cursors dotted lines 1 to 5 and ticks
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

ax1top= ax1.secondary_xaxis('top')
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels(['1', '2'])

plt.show()

table

## Q: How to reduce the chances of errors when handling variables?

**Tip**: Define the parameters, like channels and cursor positions, only once for the table and the plot.

In [ ]:
# Sampling frequency
fs = int(abf.dataPointsPerMs * 1000)

# Define channels of each signal
voltage_channel = 0
current_channel = 0

# WE CREATE A TABLE FOR THE RESULTS
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.3f}'.format

# Calculate the results for each sweep
for sweep in abf.sweepList:

    # Define channels and signals for the function
    abf.setSweep(sweep, channel=1)
    voltage_signal = abf.sweepY
    current_signal = abf.sweepC

    # HERE WE CALL THE FUNCTION
    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       fs=fs, cursor1_ms=5, cursor2_ms=500)

    # FILL THE TABLE WITH RESULTS FROM THE FUNCTION
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']

# Quick plot to visualize the plot and the cursors position
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=0)
    ax1.plot(abf.sweepX*1000, abf.sweepY)

    abf.setSweep(sweep, channel=0)
    ax2.plot(abf.sweepX*1000, abf.sweepC)

# Labels
ax1.set_ylabel(abf.sweepLabelY)
ax2.set_ylabel(abf.sweepLabelC)
ax2.set_xlabel('Time (ms)')

# Plot cursors dotted lines 1 to 5 and ticks
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

ax1top= ax1.secondary_xaxis('top')
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels(['1', '2'])

plt.show()

table

<details>
<summary><strong>Solution</strong></summary>

```python

# If you use the same parameters several times define them at the beginning
cursor1_ms = 5
cursor2_ms = 100

# Sampling frequency
fs = int(abf.dataPointsPerMs * 1000)

# Define channel
channel_no = 0

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.3f}'.format

# Calculate the results for each sweep
for sweep in abf.sweepList:

    # Define channels and signals for the function
    abf.setSweep(sweep, channel=channel_no)
    voltage_signal = abf.sweepY
    current_signal = abf.sweepC
    
    # Call the function 'passive_memb_properties'
    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       fs=fs, cursor1_ms=cursor1_ms, cursor2_ms=cursor2_ms)
    
    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    
    
# Create a list the position of cursors in ms
cursors_ms = [cursor1_ms, cursor2_ms]  # Reuse the predefined variables

# Quick plot to visualize the plot and the cursors position
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=channel_no)
    ax1.plot(abf.sweepX*1000, abf.sweepY)
        
    abf.setSweep(sweep, channel=channel_no)
    ax2.plot(abf.sweepX*1000, abf.sweepC)

# Labels
ax1.set_ylabel(abf.sweepLabelY)
ax2.set_ylabel(abf.sweepLabelC)
ax2.set_xlabel('Time (ms)')
    
# Plot cursors dotted lines 1 to 5 and ticks
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))
    
ax1top= ax1.secondary_xaxis('top')
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels(['1', '2'])

plt.show()

table
```

## Save the plot and the table

- See options for `savefig`: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html. You can define parameters like: transparent, dpi, format, etc.
- See options for `df.to_csv`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html

In [ ]:
# Save the plot
fig.savefig(f"{paths['analysis']}/{filename}.png", dpi=300)  # or svg, tif, etc.

# Save the table
table.to_csv(f"{paths['analysis']}/{filename}_passive_memb_properties.csv", index=False)

# Function passive properties: RMP + input resistance

**INPUT RESISTANCE** (Rin) is the total resistance measured by the amplifier, which is mainly the membrane resistance (Rm) + the electrode resistance. The input resistance depends on the size of the cell and the number of open ion channels. Larger cells have lower input resistance since the current will flow more easily, and cells with more open ion channels increase the conductance (1/R) of ions and, thus, also decrease the resistance. We try to measure input resistance around RMP, when the cell is in a relatively steady-state. Note that input resistance can be also calculated in voltage-clamp mode.

We can calculate input resistance using **Ohm's law**: R = V/I

We will add now the option to calculate the input resistance.


In [ ]:
def passive_memb_properties(voltage_signal, current_signal, time_signal, fs,
                            cursor3_ms, cursor4_ms, cursor5_ms,
                            cursor1_ms=5, cursor2_ms=50):

    # Index cursor position
    cursor1 = int(cursor1_ms * (fs/1000))
    cursor2 = int(cursor2_ms * (fs/1000))
    cursor3 = int(cursor3_ms * (fs/1000))
    cursor4 = int(cursor4_ms * (fs/1000))
    cursor5 = int(cursor5_ms * (fs/1000))

    # Resting membrane potential
    rmp = np.average(voltage_signal[cursor1:cursor2])

    # Voltage segment with steady voltage response
    steady_voltage = np.average(voltage_signal[cursor4:cursor5])

    # Get the current step values
    current_steps = np.average(current_signal[cursor4:cursor5])

    # Delta betwwn steady voltage response and the rmp
    voltage_delta = np.average(voltage_signal[cursor4:cursor5]) - rmp

    # Calculate input resistance
    ir = abs(voltage_delta/current_steps)*1000

    # Dictionary with results
    return {'rmp': rmp,
            'current_steps': current_steps,
            'voltage_delta': voltage_delta,
            'steady_voltage': steady_voltage,
            'ir': ir}

## Plot and table

**Q: You need to fix the following errors:**
1. Traces are not plotted
2.  Only one trace is plotted
3. Optional: add ax2 with current steps (Solution 2)

In [ ]:
# Parameters
# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Define the channel
channel_no = 0

# Define the signals
current_signal = abf.sweepC
voltage_signal = abf.sweepY

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, ax1 = plt.subplots(figsize=(5, 5))  # Figure size
plt.rcParams.update({'font.size': 12})  # Font size

# Traces
ax.set_title('Traces')
for sweep in abf.sweepList:
    voltage = abf.sweepY
    time = abf.sweepX*1000  # time in ms
    ax.plot(time, voltage)  # A: change this to ax

# Labels for cursors 1 to 5
ax1top= ax.secondary_xaxis('top')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Voltage (mV)')
ax.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines 1 to 5
for i, cursor in enumerate(cursors_ms):
    ax.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels(['1', '2', '3', '4', '5'])

for sweep in abf.sweepList:

    # Define channels and signals for the function
    abf.setSweep(sweep, channel=0)
    voltage_signal = abf.sweepY
    current_signal = abf.sweepC  # change to current signal

    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])

    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table

In [ ]:
# Parameters
# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Define the channel
channel_no = 0

# Define the signals
current_signal = abf.sweepC
voltage_signal = abf.sweepY

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, ax1 = plt.subplots(figsize=(5, 5))  # Figure size
plt.rcParams.update({'font.size': 12})  # Font size

# Traces
ax1.set_title('Traces')
for sweep in abf.sweepList:
    abf.setSweep(sweep)   # YOU NEED TO CALL THE SWEEP IN THE LOOP
    voltage = abf.sweepY
    time = abf.sweepX*1000  # time in ms
    ax1.plot(time, voltage)  # CHANGE THIS TO AX1

# Labels for cursors 1 to 5
ax1top= ax1.secondary_xaxis('top')
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Voltage (mV)')
ax1.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines 1 to 5
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels(['1', '2', '3', '4', '5'])

for sweep in abf.sweepList:

    # Define channels and signals for the function
    abf.setSweep(sweep, channel=0)
    voltage_signal = abf.sweepY
    current_signal = abf.sweepC  # change to current signal

    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])

    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table

<details>
<summary><strong>Solution</strong></summary>

```python

# Parameters
# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Define the channel
channel_no = 0

# Define the signals
current_signal = abf.sweepC
voltage_signal = abf.sweepY

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, ax1 = plt.subplots(figsize=(5, 5))  # Figure size
plt.rcParams.update({'font.size': 12})  # Font size        

# Traces
ax1.set_title('Traces')
for sweep in abf.sweepList:
    abf.setSweep(sweep)   # YOU NEED TO CALL THE SWEEP IN THE LOOP
    voltage = abf.sweepY
    time = abf.sweepX*1000  # time in ms
    ax1.plot(time, voltage)  # CHANGE THIS TO AX1

# Labels for cursors 1 to 5
ax1top= ax1.secondary_xaxis('top')
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Voltage (mV)')
ax1.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines 1 to 5
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels(['1', '2', '3', '4', '5'])

for sweep in abf.sweepList:
    
    # Define channels and signals for the function
    abf.setSweep(sweep, channel=0)
    voltage_signal = abf.sweepY
    current_signal = abf.sweepC  # change to current signal
    
    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])
    
    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table
```

<details>
<summary><strong>Solution 2</strong></summary>

```python


# Parameters
# Sampling frequency
fs = int(abf.dataPointsPerMs * 1000)

# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Define the channel
channel_no = 0

# Define the signals
current_signal = abf.sweepC
voltage_signal = abf.sweepY

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 5), sharex=True)  # Figure size
plt.rcParams.update({'font.size': 12})  # Font size        

# Traces
ax1.set_title('Traces')
for sweep in abf.sweepList:
    abf.setSweep(sweep)   # YOU NEED TO CALL THE SWEEP IN THE LOOP
    voltage = abf.sweepY
    current = abf.sweepC
    time = abf.sweepX*1000  # time in ms
    ax1.plot(time, voltage)  # CHANGE THIS TO AX1
    ax2.plot(time, current)

# Labels for cursors 1 to 5
ax1top= ax1.secondary_xaxis('top')
ax1.set_ylabel('Voltage (mV)')
ax2.set_ylabel('I (pA)')
ax2.set_xlabel('Time (ms)')
ax1.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines 1 to 5
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))
    ax2.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels(['1', '2', '3', '4', '5'])

for sweep in abf.sweepList:
    
    # Define channels and signals for the function
    abf.setSweep(sweep, channel=0)
    voltage_signal = abf.sweepY
    current_signal = abf.sweepC  # change to current signal
    
    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])
    
    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table

```

## Q: Create a second plot showing the I-V plot

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/01_I-V plot.png" width="300">

**Figure**. Current-voltage plots or I-V plots are commonly used in voltage-clamp mode to plot ion current current flow at different membrane voltages. Source: Neuroscience: Exploring the brain, Bear et al., 2015

The input resistance can be calculated as the slope of the linear ordinary least square fit (y = mx + b), x = ΔIm, y = ΔVm. You can use numpy or scipy:
- Numpy: [np.polyfit](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html) (1) + [np.polyval](https://numpy.org/doc/stable/reference/generated/numpy.polyval.html)
- Scipy: [scipy.stats.linregress](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html). Calculate a linear least-squares regression for two sets of measurements.


In [ ]:
# Parameters
# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Assign channels
voltage_channel = 0
current_channel = 0

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, ax = plt.subplots(1, 2, figsize=(8, 4))  # Here you modify the number of subplots
plt.rcParams.update({'font.size': 12})  # Font size

# Assign variables to the axes
ax_traces = ax[0]
ax_ir = ax[1]

# Traces plots
ax1.set_title('Traces')
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=voltage_channel)
    voltage = abf.sweepY
    time = abf.sweepX*1000  # time in ms
    ax_traces.plot(time, voltage)

# Labels for cursors 1 to 5
ax_traces_top= ax_traces.secondary_xaxis('top')
ax_traces.set_xlabel('Time (ms)')
ax_traces.set_ylabel('Voltage (mV)')
ax_traces.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines
for i, cursor in enumerate(cursors_ms):
    ax_traces.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax_traces.set_xticks(cursors_ms)
ax_traces_top.set_xticklabels(['1', '2', '3', '4'])

for sweep in abf.sweepList:

    # Define channels and signals for the function
    abf.setSweep(sweep, channel=voltage_channel)
    voltage_signal = abf.sweepY

    abf.setSweep(sweep, channel=current_channel)
    current_signal = abf.sweepC

    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])

    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']


# Graph: I-V fitting to calculate input resistance
current = # DEFINE THE VARIABLE
voltage = # DEFINE THE VARIABLE
voltage_list = voltage.values.tolist()
current_list = current.values.tolist()

# Linear fitting
m, b = np.polyfit(#, #, 1)  # X AND Y ARE MISSING
line = np.polyval([m, b], current)

ax_ir.plot(#, #, color='r')
ax_ir.scatter(#, #)
ax_ir.set_xlabel('Current steps (pA)')
ax_ir.set_ylabel('Voltage (mV)')

# Print the slope
print("Input resistance:", m)

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table

<details>
<summary><strong>Solution</strong></summary>

```python
# Parameters
# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Assign channels
voltage_channel = 0
current_channel = 0

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, ax = plt.subplots(1, 2, figsize=(8, 4))  # Here you modify the number of subplots
plt.rcParams.update({'font.size': 12})  # Font size

# Assign variables to the axes
ax_traces = ax[0]
ax_ir = ax[1]

# Traces plots
ax1.set_title('Traces')
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=voltage_channel)
    voltage = abf.sweepY
    time = abf.sweepX*1000  # time in ms
    ax_traces.plot(time, voltage)

# Labels for cursors 1 to 5
ax_traces_top= ax_traces.secondary_xaxis('top')
ax_traces.set_xlabel('Time (ms)')
ax_traces.set_ylabel('Voltage (mV)')
ax_traces.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines
for i, cursor in enumerate(cursors_ms):
    ax_traces.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax_traces.set_xticks(cursors_ms)
ax_traces_top.set_xticklabels(['1', '2', '3', '4'])

for sweep in abf.sweepList:
    
    # Define channels and signals for the function
    abf.setSweep(sweep, channel=voltage_channel)
    voltage_signal = abf.sweepY

    abf.setSweep(sweep, channel=current_channel)
    current_signal = abf.sweepC
    
    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])
    
    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']
        
        
# Graph: I-V fitting to calculate input resistance
current = table.loc[:, 'current_steps_pA']
voltage = table.loc[:, 'V_steady_mV']
voltage_list = voltage.values.tolist()
current_list = current.values.tolist()

# Linear fitting
m, b = np.polyfit(current_list, voltage_list, 1)
line = np.polyval([m, b], current)

ax_ir.plot(current, line, color='r')
ax_ir.scatter(current, voltage)
ax_ir.set_xlabel('Current steps (pA)')
ax_ir.set_ylabel('Voltage (mV)')

# Print the slope
print("Input resistance:", np.round(m * 1000, 1), "MOhm")

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table
```

## Q: Use scipy for linear regression

**Tip**: Use [scipy.lineregress](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html)

In [ ]:
# Parameters
# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Assign channels
voltage_channel = 0
current_channel = 0

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, ax = plt.subplots(1, 2, figsize=(8, 4))  # Here you modify the number of subplots
plt.rcParams.update({'font.size': 12})  # Font size

# Assign variables to the axes
ax_traces = ax[0]
ax_ir = ax[1]

# Traces plots
ax1.set_title('Traces')
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=voltage_channel)
    voltage = abf.sweepY
    time = abf.sweepX*1000  # time in ms
    ax_traces.plot(time, voltage)

# Labels for cursors 1 to 5
ax_traces_top= ax_traces.secondary_xaxis('top')
ax_traces.set_xlabel('Time (ms)')
ax_traces.set_ylabel('Voltage (mV)')
ax_traces.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines
for i, cursor in enumerate(cursors_ms):
    ax_traces.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax_traces.set_xticks(cursors_ms)
ax_traces_top.set_xticklabels(['1', '2', '3', '4'])

for sweep in abf.sweepList:

    # Define channels and signals for the function
    abf.setSweep(sweep, channel=voltage_channel)
    voltage_signal = abf.sweepY

    abf.setSweep(sweep, channel=current_channel)
    current_signal = abf.sweepC

    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])

    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']


# Graph: I-V fitting to calculate input resistance
current = table.loc[:, 'current_steps_pA']
voltage = table.loc[:, 'V_steady_mV']
voltage_list = voltage.values.tolist()
current_list = current.values.tolist()

# USE SCIPY LINEAR REGRESSION
# FILL THE LINES

ax_ir.plot(current, line, color='r')
ax_ir.scatter(current, voltage)
ax_ir.set_xlabel('Current steps (pA)')
ax_ir.set_ylabel('Voltage (mV)')

# Print the slope
# PRINT INPUT RESISTANCE, get the slope from results

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table

<details>
<summary><strong>Solution</strong></summary>

```python

# Parameters
# Create a list the position of cursors in ms
cursors_ms = [5, 100, 130, 280, 600]

# Assign channels
voltage_channel = 0
current_channel = 0

# Create table with results
table = pd.DataFrame()

# Number of decimals in the table
pd.options.display.float_format = '{:.4f}'.format

# Plotting (optional): general settings
fig, ax = plt.subplots(1, 2, figsize=(8, 4))  # Here you modify the number of subplots
plt.rcParams.update({'font.size': 12})  # Font size

# Assign variables to the axes
ax_traces = ax[0]
ax_ir = ax[1]

# Traces plots
ax1.set_title('Traces')
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=voltage_channel)
    voltage = abf.sweepY
    time = abf.sweepX*1000  # time in ms
    ax_traces.plot(time, voltage)

# Labels for cursors 1 to 5
ax_traces_top= ax_traces.secondary_xaxis('top')
ax_traces.set_xlabel('Time (ms)')
ax_traces.set_ylabel('Voltage (mV)')
ax_traces.set_xlim(0, 1000)   # limit the x range to display (in ms)

# Loop to plot cursor dotted lines
for i, cursor in enumerate(cursors_ms):
    ax_traces.axvline(cursor, linestyle="dotted", color='gray', label=str(i+1))

# Cursors' tick and labels
ax_traces.set_xticks(cursors_ms)
ax_traces_top.set_xticklabels(['1', '2', '3', '4'])

for sweep in abf.sweepList:
    
    # Define channels and signals for the function
    abf.setSweep(sweep, channel=voltage_channel)
    voltage_signal = abf.sweepY

    abf.setSweep(sweep, channel=current_channel)
    current_signal = abf.sweepC
    
    features = passive_memb_properties(voltage_signal=voltage_signal,
                                       current_signal=current_signal,
                                       time_signal=abf.sweepX,
                                       fs=fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])
    
    # Fill the table with the results from the function
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features ['ir']
        
        
# Graph: I-V fitting to calculate input resistance
current = table.loc[:, 'current_steps_pA']
voltage = table.loc[:, 'V_steady_mV']
voltage_list = voltage.values.tolist()
current_list = current.values.tolist()

# Linear fitting
result = linregress(current_list, voltage_list)  # returns slope, intercept, rvalue, pvalue, stderr
line = result.slope * current + result.intercept  # evaluate fitted line

ax_ir.plot(current, line, color='r')
ax_ir.scatter(current, voltage)
ax_ir.set_xlabel('Current steps (pA)')
ax_ir.set_ylabel('Voltage (mV)')

# Print the slope
print("Input resistance:", result.slope)

fig.tight_layout()  # Fit subplots within your figure cleanly

plt.show()
table

```

## Save the plots

In [ ]:
# Save the plot
fig.savefig(f"{paths['analysis']}/{filename}.png", dpi=300)  # or svg, tif, etc.

# Function passive properties: RMP + IR + capacitance + tau

**CAPACITANCE** is the ability of a biological membrane to store electrical charge, acting as a capacitor that separates charges across its thin, insulating lipid bilayer.eparating charges. The change in membrane potential is mainly characterized by a single exponential with a tau (time constant) = Rm * Cm. Tau represents the time required for the membrane potential to reach ≈63% of its final value. Thus, capacitance can be calculated as tau divided by the input resistance (Cm = tau / Rm).

For calculating the exponential decay, we can use [scipy curve_fit](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html):
- `popt`: Optimal values for the parameters such that the sum of the squared residuals of f(xdata, *popt) - ydata is minimized.
- `pcov`: The estimated approximate covariance of popt.

Additionally, we can calculate the quality of the fit by using the residual sum of squares (RSS) and the total sum of squares (TSS).
- RSS: measures unexplained variance (error).
- TSS: measures total variance in the data.

Then, we can calculate goodness of fit as: $R^2 = 1 - \frac{RSS}{TSS}$.

**Tips**:
- "You can use triple quotes (''' or \"\"\") to include multi-line comments or docstrings within a function.

In [ ]:
def passive_memb_properties(voltage_signal, current_signal, time_signal, fs,
                            cursor3_ms, cursor4_ms, cursor5_ms,
                            cursor1_ms=5, cursor2_ms=50):
    """
    Inputs:
    voltage_signal: Voltage trace, raw or pre-processed
    current_signal: Current trace
    time_signal: Time trace
    fs: Sampling frequency
    cursor3_ms: Time cursor position in ms set at the peak of the capacitive transient.
    cursor4_ms: Time cursor position in ms set at the end of the capacitive transient.
    cursor5_ms: Time cursor position in ms set at the end of the voltage step.
    cursor1_ms: Time cursor position 1 in ms set at the beginning of the trace I = 0
    cursor2_ms: Time cursor position 2 in ms set at the beginning of the trace I = 0

    Returns:
    rmp: Resting membrane potential
    current_steps: Current step values
    voltage_delta: Delta voltage response to current step between cursor 3 and 4
    steady_voltage: Steady voltage response between cursor 4 and 5
    ir: Input resistance in MOhm
    tau: Decay time constant in ms
    r_squared: Quality of the exponential fitting
    capacitance: Capacitance of the passive membrane in pF
    """

    # Position of cursors in time bins (depends on sampling frequency)
    cursor1 = int(cursor1_ms * (fs/1000))
    cursor2 = int(cursor2_ms * (fs/1000))
    cursor3 = int(cursor3_ms * (fs/1000))
    cursor4 = int(cursor4_ms * (fs/1000))
    cursor5 = int(cursor5_ms * (fs/1000))

    # steady voltage response
    steady_voltage = np.average(voltage_signal[cursor4:cursor5])

    # Resting membrane potential
    rmp = np.average(voltage_signal[cursor1:cursor2])

    # Get the current step values
    current_steps = np.average(current_signal[cursor4:cursor5])

    # Delta voltage response to current step
    voltage_delta = np.average(voltage_signal[cursor4:cursor5]) - rmp

    # Input resistance
    ir = abs(voltage_delta/current_steps)*1000

    # Calculate tau from decay time constant
    decay_voltage = voltage_signal[cursor3:cursor4]
    decay_time = (time_signal[cursor3:cursor4])*1000
    decay_voltage = decay_voltage[~np.isnan(decay_voltage)]
    decay_time = decay_time[~np.isnan(decay_time)]

    # popt: optimal values for the parameters
    # pcov: estimated covariance of popt

    # Initial estimates of the function
    popt, pcov = curve_fit(lambda t, a, tau, c: a * np.exp(-t/tau) + c,
                           xdata = decay_time,
                           ydata = decay_voltage)
    a = popt[0]
    tau = popt[1]
    c = popt[2]

    # QUALITY OF THE EXPONENTIAL FITTING
    # RSS: Residual sum of squares
    RSS = np.square(decay_voltage - (a * np.exp(-decay_time/tau) + c))
    # TSS: total sum of squares
    TSS = np.square(decay_voltage - np.mean(decay_voltage))
    r_squared = 1 - np.sum(RSS) / np.sum(TSS)

    # capacitance
    capacitance = (tau/ir) * 1000

    return {'rmp': rmp,
            'current_steps': current_steps,
            'voltage_delta': voltage_delta,
            'steady_voltage': steady_voltage,
            'ir': ir,
            'tau': tau,
            'r_squared': r_squared,
            'capacitance': capacitance,
            'a': a,
            'c': c,
            'tau': tau}

## Plot and table

In [ ]:
# Parameters
cursors_ms = [5, 100, 130, 280, 600]
cursor_indices = [int(ms * abf.dataPointsPerMs) for ms in cursors_ms]
voltage_channel = 0
current_channel = 0
table = pd.DataFrame()
pd.options.display.float_format = '{:.4f}'.format

# Initialize figure with 3 subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
plt.rcParams.update({'font.size': 12})

# Loop over sweeps
for sweep in abf.sweepList:
    # Set channels
    abf.setSweep(sweep, channel=voltage_channel)
    voltage_signal = abf.sweepY
    time_signal = abf.sweepX

    abf.setSweep(sweep, channel=current_channel)
    current_signal = abf.sweepC

    # Compute passive properties
    features = passive_memb_properties(voltage_signal, current_signal, time_signal, fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])

    # Add to table
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features['ir']
        table.loc[length, 'tau_ms'] = features['tau']
        table.loc[length, 'tau_R_squared'] = features['r_squared']
        table.loc[length, 'capacitance_pF'] = features['capacitance']

    # Plot raw trace on ax1
    ax1.plot(time_signal*1000, voltage_signal)

    # Plot voltage decay with fitted curve on ax2
    decay_voltage = voltage_signal[cursor_indices[2] : cursor_indices[3]]
    decay_time = (time_signal[cursor_indices[2]:cursor_indices[3]]) * 1000
    v_fitted = features['a'] * np.exp(-decay_time / features['tau']) + features['c']
    ax2.plot(decay_time, decay_voltage, color='gray')  # You can add alpha=0.5
    ax2.plot(decay_time, v_fitted)

# Plot cursors and labels on ax1
ax1.set_title('Raw traces')
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Voltage (mV)')
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle='dotted', color='gray', label=str(i+1))
ax1top = ax1.secondary_xaxis('top')
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels([str(i+1) for i in range(len(cursors_ms))])
ax1.set_xlim(0, 1000)

# I-V curve on ax3
current = table['current_steps_pA']
voltage = table['V_steady_mV']
m, b = np.polyfit(current.values, voltage.values, 1)
ax3.plot(current, m*current+b, color='r')
ax3.scatter(current, voltage)
ax3.set_title('I-V')
ax3.set_xlabel('Current steps (pA)')
ax3.set_ylabel('Voltage (mV)')

fig.tight_layout()
plt.show()
table

## Save the plot and table

In [ ]:
# Save the plot
fig.savefig(f"{paths['analysis']}/{filename}.png", dpi=300)  # or svg, tif, etc.

# Save the table
table.to_csv(f"{paths['analysis']}/{filename}_passive_memb_properties.csv", index=False)

## Q: How to improve the curve fitting?

**Tips**: Initial estimates improve curve fitting. These initial guesses are passed to curve_fit using the p0 argument to help the optimizer converge faster and more reliably: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html

In [ ]:
def passive_memb_properties(voltage_signal, current_signal, time_signal, fs,
                            cursor3_ms, cursor4_ms, cursor5_ms,
                            cursor1_ms=5, cursor2_ms=50):
    """
    Inputs:
    voltage_signal: Voltage trace, raw or pre-processed
    current_signal: Current trace
    time_signal: Time trace
    fs: Sampling frequency
    cursor3_ms: Time cursor position in ms set at the peak of the capacitive transient.
    cursor4_ms: Time cursor position in ms set at the end of the capacitive transient.
    cursor5_ms: Time cursor position in ms set at the end of the voltage step.
    cursor1_ms: Optional, default value is 5
    cursor2_ms: Optional, default value is 50

    Returns:
    rmp: Resting membrane potential
    current_steps: Current step values
    voltage_delta: Delta voltage response to current step between cursor 3 and 4
    steady_voltage: Steady voltage response between cursor 4 and 5
    ir: Input resistance in MOhm
    tau: Decay time constant in ms
    r_squared: Quality of the exponential fitting
    capacitance: Capacitance of the passive membrane in pF
    """

    # Position of cursors in time bins (depends on sampling frequency)
    cursor1 = int(cursor1_ms * (fs/1000))
    cursor2 = int(cursor2_ms * (fs/1000))
    cursor3 = int(cursor3_ms * (fs/1000))
    cursor4 = int(cursor4_ms * (fs/1000))
    cursor5 = int(cursor5_ms * (fs/1000))

    # steady voltage response
    steady_voltage = np.average(voltage_signal[cursor4:cursor5])

    # Resting membrane potential
    rmp = np.average(voltage_signal[cursor1:cursor2])

    # Get the current step values
    current_steps = np.average(current_signal[cursor4:cursor5])

    # Delta voltage response to current step
    voltage_delta = np.average(voltage_signal[cursor4:cursor5]) - rmp

    # Input resistance
    ir = abs(voltage_delta/current_steps)*1000

    # Calculate tau from decay time constant
    decay_voltage = voltage_signal[cursor3:cursor4]
    decay_time = (time_signal[cursor3:cursor4])*1000
    decay_voltage = decay_voltage[~np.isnan(decay_voltage)]
    decay_time = decay_time[~np.isnan(decay_time)]

    # popt: optimal values for the parameters
    # pcov: estimated covariance of popt

    # ADD INITIAL ESTIMATES (p0) to curve_fit function
    # COMPLETE THE CODE

    # Initial estimates of the function
    popt, pcov = curve_fit(lambda t, a, tau, c: a * np.exp(-t/tau) + c,
                           xdata = decay_time,
                           ydata = decay_voltage)  # INSERT P0 PARAMETERS IN THE FUNCTION

    a = popt[0]
    tau = popt[1]
    c = popt[2]

    # QUALITY OF THE EXPONENTIAL FITTING
    # RSS: Residual sum of squares
    RSS = np.square(decay_voltage - (a * np.exp(-decay_time/tau) + c))
    # TSS: total sum of squares
    TSS = np.square(decay_voltage - np.mean(decay_voltage))
    r_squared = 1 - np.sum(RSS) / np.sum(TSS)

    # capacitance
    capacitance = (tau/ir) * 1000

    return {'rmp': rmp,
            'current_steps': current_steps,
            'voltage_delta': voltage_delta,
            'steady_voltage': steady_voltage,
            'ir': ir,
            'tau': tau,
            'r_squared': r_squared,
            'capacitance': capacitance,
            'a': a,
            'c': c,
            'tau': tau}

<details>
<summary><strong>Solution</strong></summary>

```python

# Before calling curve_fit in passive_memb_properties, add initial estimates:

# Initial estimates of the function
a_initial = decay_voltage[0] - decay_voltage[-1]  # amplitude of decay
tau_initial = 20  # ms, rough guess for decay constant
c_initial = decay_voltage[-1]  # steady voltage at the end

# Initial estimates of the function    
popt, pcov = curve_fit(lambda t, a, tau, c: a * np.exp(-t/tau) + c,
                       xdata = decay_time,  
                       ydata = decay_voltage,
                       p0=[a_initial, tau_initial, c_initial])
```

## Q: Run the code again and save the plot and the table

In [ ]:
# Parameters
cursors_ms = [5, 100, 130, 280, 600]
cursor_indices = [int(ms * abf.dataPointsPerMs) for ms in cursors_ms]
voltage_channel = 0
current_channel = 0
table = pd.DataFrame()
pd.options.display.float_format = '{:.4f}'.format

# Initialize figure with 3 subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
plt.rcParams.update({'font.size': 12})

# Loop over sweeps
for sweep in abf.sweepList:
    # Set channels
    abf.setSweep(sweep, channel=voltage_channel)
    voltage_signal = abf.sweepY
    time_signal = abf.sweepX

    abf.setSweep(sweep, channel=current_channel)
    current_signal = abf.sweepC

    # Compute passive properties
    features = passive_memb_properties(voltage_signal, current_signal, time_signal, fs,
                                       cursor3_ms=cursors_ms[2],
                                       cursor4_ms=cursors_ms[3],
                                       cursor5_ms=cursors_ms[4])

    # Add to table
    length = len(table)
    table.loc[length, 'rmp_mV'] = features['rmp']
    table.loc[length, 'current_steps_pA'] = features['current_steps']
    table.loc[length, 'V_delta_mV'] = features['voltage_delta']
    table.loc[length, 'V_steady_mV'] = features['steady_voltage']
    if table.loc[length, 'current_steps_pA'] != 0:
        table.loc[length, 'Rin_MOhm'] = features['ir']
        table.loc[length, 'tau_ms'] = features['tau']
        table.loc[length, 'tau_R_squared'] = features['r_squared']
        table.loc[length, 'capacitance_pF'] = features['capacitance']

    # Plot raw trace on ax1
    ax1.plot(time_signal*1000, voltage_signal)

    # Plot voltage decay with fitted curve on ax2
    decay_voltage = voltage_signal[cursor_indices[2] : cursor_indices[3]]
    decay_time = (time_signal[cursor_indices[2]:cursor_indices[3]]) * 1000
    v_fitted = features['a'] * np.exp(-decay_time / features['tau']) + features['c']
    ax2.plot(decay_time, decay_voltage, color='gray')  # You can add alpha=0.5
    ax2.plot(decay_time, v_fitted)

# Plot cursors and labels on ax1
ax1.set_title('Raw traces')
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Voltage (mV)')
for i, cursor in enumerate(cursors_ms):
    ax1.axvline(cursor, linestyle='dotted', color='gray', label=str(i+1))
ax1top = ax1.secondary_xaxis('top')
ax1top.set_xticks(cursors_ms)
ax1top.set_xticklabels([str(i+1) for i in range(len(cursors_ms))])
ax1.set_xlim(0, 1000)

# I-V curve on ax3
current = table['current_steps_pA']
voltage = table['V_steady_mV']
m, b = np.polyfit(current.values, voltage.values, 1)
ax3.plot(current, m*current+b, color='r')
ax3.scatter(current, voltage)
ax3.set_title('I-V')
ax3.set_xlabel('Current steps (pA)')
ax3.set_ylabel('Voltage (mV)')

fig.tight_layout()
plt.show()
table